# Week 2 Interview Questions — Aliasing, Copies & Memory

Welcome to the Week 2 interview-prep notebook. The question bank on the first card is short but deep: aliasing, garbage collection, mutability, cloning, deep vs shallow copies, how nested lists and strings sit in memory, why tuples are lighter than lists, how a set decides index position, and why mutable types cannot be set/dict elements. These are the questions interviewers ask to check whether you *really* understand Python's object model — not just syntax.

The ten questions all trace back to one mental model:

- **A name is a label, an object lives in memory.** `b = a` does not copy anything; both names point to the same object. `id(obj)` returns that object's address, and `sys.getrefcount(obj)` counts how many labels point at it.
- **Immutability vs mutability decides aliasing behaviour.** For immutable objects (int, str, tuple) you cannot change the object itself, so aliases are safe — rebinding one name simply points it elsewhere. For mutable objects (list, dict, set) a change through one name is visible through all the others.
- **Copying has two levels.** `a[:]`, `list(a)`, and `.copy()` build a **shallow** copy — a new outer container whose inner (nested) objects are still shared. `copy.deepcopy(a)` rebuilds every level, giving full independence.
- **Memory explains the rest.** Lists over-allocate (hence bigger and slower), tuples store exactly what they need (smaller, faster, hashable). Sets and dicts are hash tables: position is decided by the *hash* of the item, which is why items must be immutable/hashable.

As you work through the notebook, run every snippet in the code cells: check `id()` values, watch the counts, mutate through one alias and observe the other. Once you can *predict* the output of each cell before running it, you have internalised the model. The cards below walk through aliasing, mutation, cloning, shallow and deep copies, and string interning — exactly the order of the question bank. Keep the reference-count and id examples fresh in your mind, because "why is this the same object?" is the question that ties all ten together.


| Question No. | Question |
| :--: | :-- |
| 1 | What is aliasing?  |
| 2 | What is garbage collection?  |
| 3 | What is mutability and why is it dangerous in certain scenarios? |
| 4 | What is cloning? |
| 5 | Differentiate between deep and shallow copies |
| 6 | How nested lists are stored in memory? |
| 7 | How strings are stored in memory |
| 8 | Why tuples take less memory than lists? |
| 9 | How set index position is decided? |
| 10 | Why mutable types are not allowed in sets/dicts |

## 📖 Aliasing — the same object under two names

```python
a = 4
b = a
id(a) == id(b)   # True — a and b are the SAME int object
```

- `b = a` does NOT create a new object — it sticks a second label `b` onto the object `a` already refers to.
- `id(obj)` returns an object's unique address in memory (CPython). `a` and `b` having the same `id` proves they are one object.
- For immutable types (int, str), Python may reuse an existing object with the same value — small integers and short strings are cached, which is why `id(4)` also equals `id(a)`.
- `sys.getrefcount(x)` shows how many references point to the object.
- `del a` removes only the name; the object survives as long as any reference remains (`b`, `c`, ...).
- Strings/integers with the same value frequently share one object thanks to interning — the cells below demonstrate exactly that.


In [1]:
a = 4

In [2]:
id(a)

140726039717080

In [3]:
id(4)

140726039717080

In [4]:
hex(11126784)

'0xa9c800'

In [5]:
a = 4
b = a

In [6]:
print(id(a))
print(id(b))

140726039717080
140726039717080


In [7]:
c = b

In [8]:
print(id(c))

140726039717080


In [9]:
del a

In [10]:
a

NameError: name 'a' is not defined

In [11]:
print(b)

4


In [12]:
del b
print(c)

4


In [13]:
a = 'DSMP 2022-23'
b = a
c = b

In [14]:
import sys
sys.getrefcount('DSMP 2022-23')

3

## 📖 Aliasing vs mutation

```python
L = [1, 2, 3]
L.append(4)      # MUTATION — the same list; id stays the SAME
T = (1, 2, 3)
T = T + (4,)     # tuple immutable -> rebinding creates a NEW object; id DIFFERS
a = [1, 2, 3]; b = a
b.append(4)      # mutating through b changes a too — they are the SAME list
```

- A list is **mutable**: `append` modifies the existing object in place, so its `id` does not change.
- A tuple is **immutable**: `+` cannot extend it, so Python builds a brand-new tuple and rebinds `T`. The old tuple is untouched, and any alias of it still holds the old value.
- **Alias = risk:** if two names point to the same mutable object, a change through one name shows up through the other (`b.append(4)` → `a` changes).
- Guarding pattern: pass a copy to functions that may mutate, e.g. `func(a[:])` — the cell below shows the original stays safe that way.


In [15]:
L = [1,2,3]
print(id(L))

L.append(4)
print(L)
print(id(L))

1957952801280
[1, 2, 3, 4]
1957952801280


In [16]:
T = (1,2,3)
print(id(T))

T = T + (4,)

print(T)
print(id(T))

1957952519472
(1, 2, 3, 4)
1957953707664


In [17]:
a = [1,2,3]
b = a

b.append(4)
print(b)

[1, 2, 3, 4]


In [18]:
print(a)

[1, 2, 3, 4]


In [19]:
def func(data):
  data.append(4)

a = [1,2,3]
func(a[:])
print(a)

[1, 2, 3]


## 📖 Cloning — `[:]` and `.copy()` for independent copies

```python
a = [1, 2, 3]
b = a[:]        # slice copy — a NEW list object
id(a) != id(b)  # True
b.append(4)     # a is unaffected
d = {'name': 'nitish'}
e = d.copy()    # dict has its own .copy()
```

- `a[:]`, `list(a)`, and `a.copy()` all produce a fresh top-level list clone of a 1-D list.
- `dict.copy()` clones a dictionary the same way (`b['gender'] = 'male'` does not touch `a`).
- The top level is independent: `append`ing or changing items after the copy does not affect the original.
- **But the copy is shallow** — only the outer container is new; if the list contains nested lists, those inner lists are still shared between the original and the copy. Deep copying is handled in the next two cards.


In [20]:
a = [1,2,3]
# cloning
b = a[:]

In [21]:
id(a)

1957952800768

In [22]:
id(b)

1957953127424

In [23]:
b.append(4)
b

[1, 2, 3, 4]

In [24]:
a

[1, 2, 3]

In [25]:
a = {'name':'nitish','age':33}
# aliasing->cloning
b = a.copy()

b['gender'] = 'male'

print(b)
print(a)

{'name': 'nitish', 'age': 33, 'gender': 'male'}
{'name': 'nitish', 'age': 33}


In [26]:
a = [1,2,3]
# shallow
b = a.copy()

b.append(4)

print(a)
print(b)

[1, 2, 3]
[1, 2, 3, 4]


## 📖 Shallow copy — the nested trap

```python
a = [1, 2, 3, [4, 5]]
b = a.copy()        # SHALLOW copy
id(a[-1]) == id(b[-1])   # True — the inner list is SHARED
b[-1][0] = 400      # mutating the shared inner list -> a changes too
```

- A shallow copy copies only the **outer layer**: `a` and `b` are different list objects, but their elements are the same objects.
- For a 1-D list of immutables this looks perfect — but the moment the list contains a mutable nested structure, the trap springs.
- `a[:]` is also a shallow copy; it behaves identically to `.copy()`.
- The fix when you need a fully independent structure: `copy.deepcopy(a)` — it recursively copies every nested level, as demonstrated in the next card.


In [27]:
a = [1,2,3,[4,5]]
a

[1, 2, 3, [4, 5]]

In [28]:
print(id(a[-1]))
print(id(b[-1]))

1957953047104
140726039717080


In [29]:
b = a.copy()
b

[1, 2, 3, [4, 5]]

In [30]:
print(id(a))
print(id(b))

1957953496768
1957953501696


In [31]:
b[-1][0] = 400
print(b)

[1, 2, 3, [400, 5]]


In [32]:
a

[1, 2, 3, [400, 5]]

## 📖 Deep copy — independence at every level

```python
import copy
b = copy.deepcopy(a)
id(a[-1]) != id(b[-1])   # True — the inner list is a NEW object too
b[-1][0] = 400           # a stays completely unchanged
```

- `deepcopy` builds a copy at **every** level of nesting: outer list, inner list, and everything inside.
- The result is a fully independent clone — mutating `b` never leaks into `a`, no matter how deeply nested the data is.
- Use `deepcopy` whenever your lists/dicts are nested and either the original or the copy will be modified.
- For immutable contents (e.g. strings) sharing the same object is harmless, which is why Python happily shares them even in deepcopy output — the cells demonstrate with a nested `[4, 5]` inner list that a plain `[:]` shares but `deepcopy` duplicates.


In [33]:
import copy

a = [1,2,3,[4,5]]
a

[1, 2, 3, [4, 5]]

In [34]:
b = a[:]
b

[1, 2, 3, [4, 5]]

In [35]:
b[-1][0] = 400
b

[1, 2, 3, [400, 5]]

In [36]:
a

[1, 2, 3, [400, 5]]

In [37]:
print(id(a[-1]))
print(id(b[-1]))

1957953495360
1957953495360


In [38]:
# [:] -> shallow copy

## 📖 Strings — interning

```python
s = 'hello'
id(s) == id('hello')  # same object often
```

- Strings immutable → reused in memory
- This "why same id" question comes up often
- Syntax: `{12,}` — set singleton
- Week 2 = a treasure-trove of pointers + copies

In [39]:
s = 'hello'

id(s)

1957949014976

In [40]:
id(s[0])

140726039784008

In [41]:
id('h')

140726039784008

In [42]:
{12,}

{12}

## Summary

This notebook answered the ten Week 2 questions about Python's object model. You learned that **alias** means two names for one object — `b = a` shares the same object, confirmed by identical `id()` values and `sys.getrefcount`. Immutable types (ints, strings) are often interned/reused, so equal values can share an id; mutable types change in place, so aliasing a list makes every name see the mutation. **Cloning** creates independent top-level objects: `a[:]`, `list(a)`, and `.copy()` work but are **shallow** — the inner, nested containers are still shared (the nested-list trap). `copy.deepcopy()` rebuilds every level for full independence. **Memory**: lists over-allocate buffer space so they are larger and slower than tuples, which are fixed-size, lighter, faster, and hashable; that is also why tuples (not lists) can be dict keys or set members. Sets and dicts are hash tables, so element *position* is decided by hash value, which is why they demand immutable, hashable items. Keep the table in mind — mutable vs immutable, shallow vs deep — and these questions become easy.
